In [7]:
import math
import os
import sys
from pathlib import Path

import einops
import numpy as np
import torch as t
from torch import Tensor

# Make sure exercises are in the path.
# Walk up from the notebook's cwd until we find the ARENA/ folder that
# contains ARENA_3.0 — this way the notebook keeps working no matter how
# deep it's nested under scratchpad_notebooks/ (e.g. scratchpad_notebooks/1/).
def find_arena_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ARENA_3.0").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find an ARENA_3.0 folder above {start}")

root_dir = find_arena_root(Path.cwd())
exercises_dir = root_dir / "ARENA_3.0" / "chapter0_fundamentals" / "exercises"
section_dir = exercises_dir / "part0_prereqs"

print(exercises_dir)
print(exercises_dir.exists())         # should be True now
print(section_dir.exists())           # should be True now

if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))


import part0_prereqs.tests as tests
from part0_prereqs.utils import display_array_as_img, display_soln_array_as_img

MAIN = __name__ == "__main__"

/Users/shubmeister/Desktop/ARENA/ARENA_3.0/chapter0_fundamentals/exercises
True
True


In [54]:
arr = np.load(section_dir / "numbers.npy")

In [9]:
arr.shape

(6, 3, 150, 150)

In [10]:
print(arr[0].shape)
display_array_as_img(arr[0])  # plotting the first image in the batch

(3, 150, 150)


In [11]:
print(arr[0, 0].shape)
display_array_as_img(arr[0, 0])  # plotting the first channel of the first image, as monochrome

(150, 150)


In [12]:
arr_stacked = einops.rearrange(arr, "b c h w -> c h (b w)")
print(arr_stacked.shape)
display_array_as_img(arr_stacked) 

(3, 150, 900)


In [13]:
arr.shape

(6, 3, 150, 150)

In [14]:
arr1 = einops.rearrange(arr, "b c h w -> c (b h) w")
print(arr1.shape)
display_array_as_img(arr1) 

(3, 900, 150)


In [ ]:
arr2 = einops.repeat(arr[0], "c h w -> c (2 h) w")
print(arr2.shape)
display_array_as_img(arr2) 

(3, 300, 150)


In [33]:
arr[:2].shape

(2, 3, 150, 150)

In [36]:
arr3 = einops.rearrange(arr[:2], "b c h w -> c (b h) w")
print(arr3.shape)
display_array_as_img(arr3) 

(3, 300, 150)


In [37]:
arr4 = einops.repeat(arr[0], "c h w -> c (h 2) w")
print(arr4.shape)
display_array_as_img(arr4)

(3, 300, 150)


In [51]:
arr5 = einops.repeat(arr[:2], "c h w -> h (c w)")
print(arr5.shape)
display_array_as_img(arr5)

(150, 300)


In [58]:
arr6 = einops.rearrange(arr, "(b1 b2) c h w -> c (b1 h) (b2 w)", b1=2)
print(arr6.shape)
display_array_as_img(arr6)

(3, 300, 450)


In [61]:
arr7 = einops.rearrange(arr[1], "c h w -> c w h")
arr7.shape

(3, 150, 150)

In [62]:
arr8 = einops.reduce(arr, "(b1 b2) c (h h2) (w w2) -> c (b1 h) (b2 w)", "max", h2=2, w2=2, b1=2)

display_array_as_img(arr8)

In [63]:
x = t.ones((3, 1, 5))
y = t.ones((1, 4, 5))

z = x + y

In [ ]:
x = t.ones((8, 2, 6))
y = t.ones((8, 2))

z = x + y

In [66]:
def assert_all_equal(actual: Tensor, expected: Tensor) -> None:
    assert actual.shape == expected.shape, f"Shape mismatch, got: {actual.shape}"
    assert (actual == expected).all(), f"Value mismatch, got: {actual}"
    print("Tests passed!")


def assert_all_close(actual: Tensor, expected: Tensor, atol=1e-3) -> None:
    assert actual.shape == expected.shape, f"Shape mismatch, got: {actual.shape}"
    t.testing.assert_close(actual, expected, atol=atol, rtol=0.0)
    print("Tests passed!")

In [76]:
    
def rearrange_1() -> Tensor:
    """Return the following tensor using only t.arange and einops.rearrange:

    [[3, 4],
     [5, 6],
     [7, 8]]
    """
    return einops.rearrange(np.arange(3, 9), "(h w) -> h w", h=3, w=2)


expected = t.tensor([[3, 4], [5, 6], [7, 8]])
assert_all_equal(rearrange_1(), expected)

Tests passed!


In [79]:
def rearrange_2() -> Tensor:
    """Return the following tensor using only t.arange and einops.rearrange:

    [[1, 2, 3],
     [4, 5, 6]]
    """
    return einops.rearrange(np.arange(1, 7), "(h w) -> h w", h=2, w=3)


assert_all_equal(rearrange_2(), t.tensor([[1, 2, 3], [4, 5, 6]]))

Tests passed!


In [84]:
def temperatures_average(temps: Tensor) -> Tensor:
    """Return the average temperature for each week.

    temps: a 1D temperature containing temperatures for each day.
    Length will be a multiple of 7 and the first 7 days are for the first week, second 7 days for the second week, etc.

    You can do this with a single call to reduce.
    """
    assert len(temps) % 7 == 0
    
    return einops.reduce(temps, "(h 7) -> h", "mean")


temps = t.tensor([71, 72, 70, 75, 71, 72, 70, 75, 80, 85, 80, 78, 72, 83]).float()
expected = [71.571, 79.0]
assert_all_close(temperatures_average(temps), t.tensor(expected))

Tests passed!


In [92]:
def temperatures_differences(temps: Tensor) -> Tensor:
    """For each day, subtract the average for the week the day belongs to.

    temps: as above
    """
    assert len(temps) % 7 == 0
    avg = einops.reduce(temps, "(w 7) -> w", "mean")
    return temps - einops.repeat(avg, "w -> (w 7)")


expected = [-0.571, 0.429, -1.571, 3.429, -0.571, 0.429, -1.571, -4.0, 1.0, 6.0, 1.0, -1.0, -7.0, 4.0]
actual = temperatures_differences(temps)
assert_all_close(actual, t.tensor(expected))

Tests passed!


In [96]:
def temperatures_normalized(temps: Tensor) -> Tensor:
    """For each day, subtract the weekly average and divide by the weekly standard deviation.

    temps: as above

    Pass t.std to reduce.
    """
    
    averages = einops.reduce(temps, "(w 7) -> w", "mean")
    stds = einops.reduce(temps, "(h 7) -> h", t.std)
    return (temps - einops.repeat(averages, "w -> (w 7)")) / einops.repeat(stds, "w -> (w 7)")


expected = [-0.333, 0.249, -0.915, 1.995, -0.333, 0.249, -0.915, -0.894, 0.224, 1.342, 0.224, -0.224, -1.565, 0.894]
actual = temperatures_normalized(temps)
assert_all_close(actual, t.tensor(expected))

Tests passed!


In [123]:
def normalize_rows(matrix: Tensor) -> Tensor:
    """Normalize each row of the given 2D matrix.

    matrix: a 2D tensor of shape (m, n).

    Returns: a tensor of the same shape where each row is divided by its l2 norm.
    """
    mat = matrix.norm(keepdim=True, dim=1)
    return matrix / mat


matrix = t.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]).float()
expected = t.tensor([[0.267, 0.535, 0.802], [0.456, 0.570, 0.684], [0.503, 0.574, 0.646]])
assert_all_close(normalize_rows(matrix), expected)

Tests passed!


In [129]:
def cos_sim_matrix(matrix_3_3: Tensor) -> Tensor:
    """Return the cosine similarity matrix for each pair of rows of the given matrix.

    matrix: shape (m, n)
    """
    
    matrix_3_3 = normalize_rows(matrix_3_3)        
    return matrix_3_3 @ matrix_3_3.T


matrix = t.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]).float()
expected = t.tensor([[1.0, 0.975, 0.959], [0.975, 1.0, 0.998], [0.959, 0.998, 1.0]])
assert_all_close(cos_sim_matrix(matrix), expected)

Tests passed!


In [131]:
def sample_distribution(probs: Tensor, n: int) -> Tensor:
    """Return n random samples from probs, where probs is a normalized probability distribution.

    probs: shape (k,) where probs[i] is the probability of event i occurring.
    n: number of random samples

    Return: shape (n,) where out[i] is an integer indicating which event was sampled.

    Use t.rand and t.cumsum to do this without any explicit loops.
    """
    
    print(probs, probs.shape)


n = 5_000_000
probs = t.tensor([0.05, 0.1, 0.1, 0.2, 0.15, 0.4])
freqs = t.bincount(sample_distribution(probs, n)) / n
assert_all_close(freqs, probs)

tensor([0.0500, 0.1000, 0.1000, 0.2000, 0.1500, 0.4000]) torch.Size([6])


TypeError: bincount(): argument 'input' (position 1) must be Tensor, not NoneType